In [6]:
import json

with open("experimental_outputs/label_change_intervention_results.json", "r") as f:
    data = json.load(f)

VALID_SUBSETS = {"true", "false"}
VALID_STRENGTHS = {-5, -2, -1, 0, 1, 2, 5}
VALID_LAYERS = {9, 10, 11, 12, 13}
VALID_MODELS = {'llama-3.1-8B-Instruct', 'llama-3.1-8B'}

# --------------------------------------------------
# FILTER + BUILD DICTIONARY
# --------------------------------------------------

result = {}

for d in data:
    
    # filter subset
    if d.get("model") not in VALID_MODELS:
        continue

    strength = d.get("intervention")
    model = d["model"]
    probe = d["probe class"]
    subset = d["subset"]
    p_diff = d["p_diff"]

    # train_datasets is a list → make it hashable & order-invariant
    train_dataset = "+".join(sorted(d["train_datasets"]))

    key = (model, probe, train_dataset, subset, strength)
    result[key] = p_diff

# --------------------------------------------------
# RESULT
# --------------------------------------------------

print(f"Number of entries: {len(result)}")


Number of entries: 96


In [10]:
import pandas as pd
from collections import defaultdict
strength = 5  # you can loop over this later if needed
rows = []

# get unique dimensions from result keys
models = sorted({k[0] for k in result})
probes = sorted({k[1] for k in result})
train_datasets = sorted({k[2] for k in result})

for probe in probes:
    for train_dataset in train_datasets:
        row = {
            "probe": probe,
            "train_dataset": train_dataset,
        }

        for model in models:
            try:
                # baseline terms
                f0 = result[(model, probe, train_dataset, "false", 0)]
                t0 = result[(model, probe, train_dataset, "true", 0)]

                # strength terms
                f_pos = result[(model, probe, train_dataset, "false", strength)]
                t_neg = result[(model, probe, train_dataset, "true", -strength)]

                NIE_false_to_true = (f_pos - f0) / (t0 - f0)
                NIE_true_to_false = (t_neg - t0) / (f0 - t0)

                row[(model, "false→true")] = NIE_false_to_true
                row[(model, "true→false")] = NIE_true_to_false

            except KeyError:
                # missing data → leave as NaN
                row[(model, "false→true")] = float("nan")
                row[(model, "true→false")] = float("nan")

        rows.append(row)
df = pd.DataFrame(rows)
df = df.set_index(["probe", "train_dataset"])
df = df.round(2)
df

(llama-3.1-8B, false→true)  \
probe   train_dataset                                          
LRProbe cities                                          0.05   
        larger_than                                     0.06   
        larger_than+smaller_than                        0.04   
MMProbe cities                                          0.13   
        larger_than                                     0.11   
        larger_than+smaller_than                        0.13   

                                  (llama-3.1-8B, true→false)  \
probe   train_dataset                                          
LRProbe cities                                          0.01   
        larger_than                                     0.06   
        larger_than+smaller_than                        0.04   
MMProbe cities                                          0.38   
        larger_than                                     0.30   
        larger_than+smaller_than                        0.25   

                                  (llama-3.1-8B-Instruct, false→true)  \
probe   train_dataset                                                   
LRProbe cities                                                   0.04   
        larger_than                                              0.11   
        larger_than+smaller_than                                 0.06   
MMProbe cities                                                   0.04   
        larger_than                                              0.12   
        larger_than+smaller_than                                 0.16   

                                  (llama-3.1-8B-Instruct, true→false)  
probe   train_dataset                                                  
LRProbe cities                                                   0.12  
        larger_than                                              0.80  
        larger_than+smaller_than                                 0.54  
MMProbe cities                                                   0.71  
        larger_than                                              0.97  
        larger_than+smaller_than                                 0.95

In [1]:
import torch as t
import pandas as pd
import os
from tqdm import tqdm
from utils import collect_acts
from generate_acts import load_model
from probes import LRProbe, MMProbe, CCSProbe
import plotly.express as px
import json
import argparse
import configparser

DEBUG = False
if DEBUG:
    tracer_kwargs = {'scan': True, 'validate': True}
else:
    tracer_kwargs = {'scan': False, 'validate': False}

def intervention_experiment(model, queries, direction, strength, hidden_states, batch_size=5, remote=True):
    """
    model : an nnsight LanguageModel
    queries : a list of statements to be labeled
    direction : a direction in the residual stream of the model
    hidden_states : list of (layer, -1 or 0) pairs, -1 for intervene before the period, 0 for intervene over the period
    subtract : if True, subtract the direction from the hidden states instead of adding it
    batch_size : batch size for forward passes
    remote : run on the NDIF server?
    Add the direction to the specified hidden states and return the resulting probability diff P(TRUE) - P(FALSE)
    and sum P(TRUE) + P(FALSE) averaged over the data
    """

    true_idx, false_idx = model.tokenizer.encode(' TRUE')[-1], model.tokenizer.encode(' FALSE')[-1]
    len_suffix = len(model.tokenizer.encode('This statement is:'))

    p_diffs = []
    tots = []
    for batch_idx in range(0, len(queries), batch_size):
        batch = queries[batch_idx:batch_idx+batch_size]
        for layer, offset in hidden_states:
            with t.no_grad():
                with model.trace(batch, remote=remote, **tracer_kwargs):
                    model.model.layers[layer].output[:,-len_suffix + offset, :] += \
                        direction * strength
                    logits = model.lm_head.output[:, -1, :]
                    probs = logits.softmax(-1)
                    p_diffs.append((probs[:, true_idx] - probs[:, false_idx]).save())
                    tots.append((probs[:, true_idx] + probs[:, false_idx]).save())
    p_diffs = t.cat([p_diff for p_diff in p_diffs])
    tots = t.cat([tot for tot in tots])

    return p_diffs.mean().item(), tots.mean().item()

def get_probe_direction(ProbeClass, model_name, train_datasets, end_layer, noperiod):
    print(f'training {model_name} probe...')
    if ProbeClass == LRProbe or ProbeClass == MMProbe or ProbeClass == 'random':
        acts, labels = [], []
        for dataset in train_datasets:
            acts.append(collect_acts(dataset, model_name, end_layer, noperiod=noperiod).to('cuda:0'))
            labels.append(t.Tensor(pd.read_csv(f'datasets/{dataset}.csv')['label'].tolist()).to('cuda:0'))
        acts, labels = t.cat(acts), t.cat(labels)
        if ProbeClass == LRProbe or ProbeClass == MMProbe:
            probe = ProbeClass.from_data(acts, labels, device='cuda:0')
        elif ProbeClass == 'random':
            probe = MMProbe.from_data(acts, labels, device='cuda:0')
            probe.direction = t.nn.Parameter(t.randn_like(probe.direction))
    elif ProbeClass == CCSProbe:
        acts = collect_acts(train_datasets[0], model_name, end_layer, noperiod=noperiod).to('cuda:0')
        neg_acts = collect_acts(train_datasets[1], model_name, end_layer, noperiod=noperiod).to('cuda:0')
        labels = t.Tensor(pd.read_csv(f'datasets/{train_datasets[0]}.csv')['label'].tolist()).to('cuda:0')
        probe = ProbeClass.from_data(acts, neg_acts, labels=labels, device='cuda:0')

    direction = probe.direction
    probe_direction = direction.clone().cuda()
    true_acts, false_acts = acts[labels==1], acts[labels==0]
    true_mean, false_mean = true_acts.mean(0), false_acts.mean(0)
    direction = direction / direction.norm()
    diff = (true_mean - false_mean) @ direction
    direction = diff * direction
    # direction is from true to false
    direction = direction.cuda()
    return probe_direction, direction

def prepare_data(prompt, dataset, subset='all'):
    """
    prompt : the few shot prompt
    dataset : dataset name
    model : an nnsight LanguageModel
    subset : 'all', 'true', or 'false'
    Returns a list of queries to be run through the model for the patching experiment
    and a list of the index of the last period token in each query.
    """
    df = pd.read_csv(f'datasets/{dataset}.csv')
    if subset == 'all':
        statements = df['statement'].tolist()
    elif subset == 'true':
        statements = df[df['label'] == 1]['statement'].tolist()
    elif subset == 'false':
        statements = df[df['label'] == 0]['statement'].tolist()

    queries = []
    for statement in statements:
        if statement not in prompt:
            queries.append(prompt + statement + ' This statement is:')
    
    return queries

if __name__ == '__main__':
    args = {
        'model_1': 'llama-3.2-3B',
        'model_2': 'llama-3.2-3B-Instruct',
        'probe': 'MMProbe',
        'device': 'cuda',
        'noperiod': 'false',
    }

    remote = args['device'] == 'remote'

    # prepare hidden states to intervene over
    config = configparser.ConfigParser()
    config.read('config.ini')
    end_layer = 13
    noperiod = False
    for train_set in [['cities'], ["cities", "neg_cities"], ['larger_than'], ['larger_than', 'smaller_than']]:
        print(f"train set {train_set}")
        probe_direction_1, direction_1 = get_probe_direction(eval(args['probe']), args['model_1'], train_set, end_layer, noperiod)
        probe_direction_2, direction_2 = get_probe_direction(eval(args['probe']), args['model_2'], train_set, end_layer, noperiod)

        cos_sim = t.dot(direction_1, direction_2) / (
                    t.norm(direction_1) * t.norm(direction_2)
                )
        raw_probe_cos_sim = t.dot(probe_direction_1, probe_direction_2) / (
                    t.norm(direction_1) * t.norm(direction_2)
                )
        print(f"{train_set}: {cos_sim}")

/venv/geometry-of-truth/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


train set ['cities']
training llama-3.2-3B probe...
training llama-3.2-3B-Instruct probe...
['cities']: 0.48188185691833496
train set ['cities', 'neg_cities']
training llama-3.2-3B probe...
training llama-3.2-3B-Instruct probe...
['cities', 'neg_cities']: 0.566983699798584
train set ['larger_than']
training llama-3.2-3B probe...
training llama-3.2-3B-Instruct probe...
['larger_than']: 0.5736063718795776
train set ['larger_than', 'smaller_than']
training llama-3.2-3B probe...
training llama-3.2-3B-Instruct probe...
['larger_than', 'smaller_than']: 0.6051793098449707
